In [1]:
#Data Preparation
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import skipgrams
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras import backend as K

In [2]:
# Sample text
corpus = ["the quick brown fox jumped over the lazy dog"]

# Tokenize words
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word2id = tokenizer.word_index
id2word = {v: k for k, v in word2id.items()}

vocab_size = len(word2id) + 1
print("Vocabulary:", word2id)

# Convert text to sequence of word IDs
seq = tokenizer.texts_to_sequences(corpus)[0]
print("\nText to sequence:", seq)

Vocabulary: {'the': 1, 'quick': 2, 'brown': 3, 'fox': 4, 'jumped': 5, 'over': 6, 'lazy': 7, 'dog': 8}

Text to sequence: [1, 2, 3, 4, 5, 6, 1, 7, 8]


In [6]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import skipgrams

window_size = 2
# Provide a seed value to avoid internal float-to-int error
seed_value = 42

# Generate skipgram pairs and labels
pairs, labels = skipgrams(sequence=seq, vocabulary_size=vocab_size, window_size=window_size, seed=seed_value)

print("\nSample context-target pairs (first 5):")
for i in range(5):
    print(f"Context: {id2word[pairs[i][0]]}, Target: {id2word[pairs[i][1]]}, Label: {labels[i]}")

# Convert to NumPy arrays
pairs = np.array(pairs)
labels = np.array(labels)


Sample context-target pairs (first 5):
Context: lazy, Target: the, Label: 0
Context: the, Target: lazy, Label: 1
Context: brown, Target: lazy, Label: 0
Context: fox, Target: jumped, Label: 1
Context: jumped, Target: the, Label: 1


In [7]:
#Build and Train the CBOW Model
embed_dim = 8  # Size of embedding vector

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=2))
model.add(Lambda(lambda x: K.mean(x, axis=1)))  # CBOW averages context embeddings
model.add(Dense(vocab_size, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
print("\nTraining CBOW model...")
model.fit(pairs, labels, epochs=200, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Training CBOW model...


In [8]:
#Output: Word Embeddings and Predictions
weights = model.get_layer('embedding').get_weights()[0]
print("\nWord Embedding shape:", weights.shape)

# Display embeddings for each word
for word, idx in word2id.items():
    print(f"{word}: {weights[idx]}")

# Test prediction
test_context = np.array([[word2id['quick'], word2id['fox']]])
pred = model.predict(test_context)
predicted_word = id2word[np.argmax(pred)]
print("\nPredicted word for context ['quick', 'fox'] →", predicted_word)


Word Embedding shape: (9, 8)
the: [-0.7346975  -0.6703692  -0.4688449   0.8485153  -0.15938595 -0.0754217
  0.33654252  0.8514567 ]
quick: [-0.59336483 -0.519877   -0.35434532  0.6155105  -0.15968914 -0.02092001
  0.32355925  0.5830887 ]
brown: [-0.7125971  -0.5423697  -0.25781712  0.69785434 -0.27987134 -0.20399351
  0.50079286  0.6903903 ]
fox: [-0.78783584 -0.46256685 -0.17758064  0.7331867  -0.4310711  -0.31476048
  0.51055086  0.77004933]
jumped: [-0.6844988  -0.521051   -0.23220478  0.7290944  -0.30725563 -0.23207927
  0.4286127   0.7370967 ]
over: [-0.691306   -0.5169836  -0.25955364  0.6955762  -0.27597743 -0.20381805
  0.47235262  0.7525323 ]
lazy: [-0.659473   -0.40650508 -0.10575949  0.68024164 -0.36763462 -0.31199178
  0.49148825  0.68596715]
dog: [-0.5920329  -0.36451992 -0.06616377  0.5828247  -0.34124944 -0.3649716
  0.5277296   0.6250173 ]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step

Predicted word for context ['quick', 'fox'] → the
